MTH 4320 / 5320 - Homework 1

Multivariate Linear Regression and Gradient Descent

Name: Jacob Russ
Course: MTH 4320 / 5320 - Deep Learning
Assignment: Homework 1

## Problem 1 - Multivariate Linear Regression Model and Loss





### 1.1 Model Notation

Let: 
- n be the number of training examples
- d be the number of input features
- m be the number of output variables 

Let $X_{\text{original}}$ be the input matrix, where $X_{\text{original}} \in \mathbb{R}^{n\times d}$. We incoporate the bias/intercept term by adding a column of ones to $X_{\text{original}}$, and we get the following matrix. 

$$
X = 
\begin{bmatrix}
1 & x_{11} & x_{12} & \cdots & x_{1d}\\
1 & x_{21} & x_{22} & \cdots & x_{2d}\\
\vdots & \vdots & \vdots & \ddots & \vdots\\
1 & x_{n1} & x_{n2} & \cdots & x_{nd}
\end{bmatrix}$$

Therefore,

$X \in \mathbb{R}^{n\times(d+1)}$.

Let Y be the matrix of target outputs:

$Y \in \mathbb{R}^{n\times m}$.

Because the model has m output variables, the weight matrix is:

$W \in \mathbb{R}^{(d+1)\times m}$.




### 1.2 Model Prediction

The predicted output matrix is given by


$\hat{Y} = XW$.


The dimensions are


$X \in \mathbb{R}^{n\times(d+1)}$


and


$W \in \mathbb{R}^{(d+1)\times m}$.


Therefore, the resulting prediction matrix has dimensions


$\hat{Y} \in \mathbb{R}^{n\times m}$.


The matrix multiplication is valid because the inner dimensions match:


$(n\times(d+1))((d+1)\times m)=n\times m$.





### 1.3 Mean Squared Error Loss

The mean squared error (MSE) measures the average squared difference between the actual outputs Y and the predicted outputs $\hat{Y}$.

For n observations and m output variables, the MSE loss is

$$ 
L(W)=\frac{1}{nm}\sum_{i=1}^{n}\sum_{j=1}^{m}
\left(Y_{ij}-\hat{Y}_{ij}\right)^2
$$

Using the prediction equation $\hat{Y}=XW$, this can also be written as

$$
L(W)=\frac{1}{nm}\sum_{i=1}^{n}\sum_{j=1}^{m}
\left(Y_{ij}-(XW)_{ij}\right)^2
$$

The error matrix is

E = Y-$\hat{Y}$ = Y-XW,

with dimensions

$E\in\mathbb{R}^{n\times m}$.

The MSE loss is a scalar value:

$L(W)\in\mathbb{R}$.

---

## Problem 2 - Gradient of the MSE Loss

From Problem 1, the MSE loss is

$$
L(W)=\frac{1}{nm}\sum_{i=1}^{n}\sum_{j=1}^{m}
\left(Y_{ij}-(XW)_{ij}\right)^2.
$$

In order to find the gradient, we must find the partial derivatives and collect them to create the full gradient matrix.

We can first find the partial derivative with respect to one weight, $W_{kj}$. Using the chain rule and power rule,

$$
\frac{\partial L}{\partial W_{kj}} =

\frac{1}{nm}
\sum_{i=1}^{n}
2\left(Y_{ij}-(XW)_{ij}\right)
\frac{\partial}{\partial W_{kj}}
\left(Y_{ij}-(XW)_{ij}\right).
$$

The target value $Y_{ij}$ does not depend on $W$, so its derivative is zero. Also, the $(i,j)$-th element of $XW$ can be written as

$$
(XW)_{ij} =

\sum_{k=0}^{d}X_{ik}W_{kj}.
$$

Therefore,

$$
\frac{\partial (XW)_{ij}}{\partial W_{kj}}

= X_{ik}.
$$

Because the error contains $-(XW)_{ij}$, we have

$$
\frac{\partial}{\partial W_{kj}}
\left(Y_{ij}-(XW)_{ij}\right) =

-X_{ik}.
$$

Substituting this result gives

$$
\frac{\partial L}{\partial W_{kj}} =

-\frac{2}{nm}
\sum_{i=1}^{n}
X_{ik}\left(Y_{ij}-(XW)_{ij}\right).
$$

Equivalently,

$$
\frac{\partial L}{\partial W_{kj}} =

\frac{2}{nm}
\sum_{i=1}^{n}
X_{ik}\left((XW)_{ij}-Y_{ij}\right).
$$

This gives the derivative of the loss with respect to an individual element of $W$. The next step is to collect these partial derivatives into the full gradient matrix, $\nabla_W L$.

## Collect the Partial Derivatives

The ((k,j))-th element of the gradient is

$$ \frac{\partial L}{\partial W_{kj}} =

\frac{2}{nm}
\sum_{i=1}^{n}
X_{ik}\left((XW)_{ij}-Y_{ij}\right).
$$

The summation over (i) is the ((k,j))-th element of the matrix product

$$
X^T(XW-Y).
$$

Therefore, collecting all of the partial derivatives into the gradient matrix gives

$$ \nabla_W L =

\frac{2}{nm}X^T(XW-Y).
$$

The dimensions are

$$
X^T\in\mathbb{R}^{(d+1)\times n}
$$

and

$$
XW-Y\in\mathbb{R}^{n\times m}.
$$

Therefore,

$$
X^T(XW-Y)\in\mathbb{R}^{(d+1)\times m},
$$

which matches the dimensions of (W).

Thus, the final gradient is

$$ \boxed{ \nabla_W L =

\frac{2}{nm}X^T(XW-Y)
}
$$

---

## Problem 3 - Implementation of Multivariate Linear Regression with Gradient Descent


In [9]:
import numpy as np

class MultivariateLinearRegressionGD:
    def __init__(self, learning_rate=1e-2, max_iter=1000, tolerance=1e-8):
        # store hyperparameters here
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.tolerance = tolerance
        # randomly initialize weights
        self.coef_ = None
        self.loss_history = []

    def fit(self, X, Y):
        # validate inputs
        X = np.asarray(X, dtype=float)
        Y = np.asarray(Y, dtype=float)
        if X.ndim != 2: 
            raise ValueError("X must be a 2D array.")
        
        if Y.ndim == 1:
         Y = Y.reshape(-1, 1)

        if X.shape[0] != Y.shape[0]:
             raise ValueError("X and Y must contain the same number of samples.")
        
        # add the bias column
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        
        # initialize weights
        self.coef_ = np.random.normal(0, 0.01, X.shape[1])
        
        # run gradient descent
        for i in range(self.max_iter):
            # compute predictions
            Y_pred = X @ self.coef_
            # compute loss
            error = Y_pred - Y
            loss = np.mean(error) ** 2
            self.loss_history.append(loss)
            # compute gradients
            gradients = (2 / X.shape[0]) * (X.T @ (error))
            # check for convergence
            if np.linalg.norm(gradients) < self.tolerance:
                break
            # update weights
            self.coef_ -= self.learning_rate * gradients

            # store learned parameters as attributes
            return self

def predict(self, X):
        # add the bias column
        X = np.asarray(X, dtype=float)
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        #return model predictions
        return X @ self.coef_

---

## Problem 4 - Multiple Random Initializations

In [ ]:
import numpy as np

class MultivariateLinearRegressionGD:
    def __init__(self, learning_rate=1e-2, max_iter=1000, tolerance=1e-8, n_initializations=5, random_state=None):
        # store hyperparameters here
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.tolerance = tolerance
        self.n_initializations = n_initializations
        self.random_state = random_state
        # randomly initialize weights
        self.coef_ = None
        self.loss_history = []
        self.validation_loss_history = []
        self.selected_initialization = None

    def fit(self, X, X_Val, Y, Y_Val):
        # validate inputs
        X = np.asarray(X, dtype=float)
        X_Val = np.asarray(X_Val, dtype=float)
        Y = np.asarray(Y, dtype=float)
        Y_Val = np.asarray(Y_Val, dtype=float)

        if X.ndim != 2 or X_Val.ndim != 2: 
            raise ValueError("X and X_Val must be 2D arrays.")
        
        if Y.ndim == 1:
         Y = Y.reshape(-1, 1)

        if Y_Val.ndim == 1:
         Y_Val = Y_Val.reshape(-1, 1)
        
        if X.shape[0] != Y.shape[0]:
             raise ValueError("X and Y must contain the same number of samples.")
        
        if X_Val.shape[0] != Y_Val.shape[0]:
             raise ValueError("X_Val and Y_Val must contain the same number of samples.")
        if X.shape[1] != X_Val.shape[1]:
             raise ValueError("Training and validation sets must have the same number of features.")

        #reset histories
        self.loss_history = []
        self.validation_loss_history = []
        self.selected_initialization = None
        
        # add the bias column
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        X_Val = np.hstack((np.ones((X_Val.shape[0], 1)), X_Val))

        # create a random number generator
        rng = np.random.default_rng(self.random_state) 

        best_mse = np.inf
        best_weights = None

        for init in range(self.n_initializations):
          # initialize weights
          self.coef_ = rng.normal(0, 0.01, (X.shape[1], Y.shape[1]))
        
          # run gradient descent
          for i in range(self.max_iter):
            # compute predictions
            Y_pred = X @ self.coef_
            # compute loss
            error = Y_pred - Y
            loss = np.mean(error ** 2)
            self.loss_history.append(loss)
            # compute gradients
            gradients = (2 / X.shape[0]) * (X.T @ (error))
            # check for convergence
            if np.linalg.norm(gradients) < self.tolerance:
                break
            # update weights
            self.coef_ -= self.learning_rate * gradients

          #Compute validation performance
          Y_Val_pred = X_Val @ self.coef_
          validation_mse = np.mean((Y_Val_pred - Y_Val) ** 2)
          self.validation_loss_history.append(validation_mse)

          # Check if this initialization is the best so far
          if validation_mse < best_mse:
            best_mse = validation_mse
            best_weights = self.coef_.copy()
            self.selected_initialization = init

        # store best weights
        self.coef_ = best_weights
        return self
    
    def report_validation_performance(self):
            #report validation MSE for each initialization 
            for i, mse in enumerate(self.validation_loss_history):
                print(f"Initialization {i+1}: Validation MSE = {mse:.10f}")
            #report 
            print(f"\nSelected Initialization: {self.selected_initialization+1} with Validation MSE = {self.validation_loss_history[self.selected_initialization]:.10f}")


    def predict(self, X):
        # add the bias column
        X = np.asarray(X, dtype=float)
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        #return model predictions
        return X @ self.coef_

---

## Problem 5 - Applying Model to the Linnerud Dataset

In [33]:
# Problem 5 - Apply Model to Linnerud Dataset

from sklearn.datasets import load_linnerud
from sklearn.model_selection import train_test_split

data = load_linnerud()

# Split into training, validation, and test sets

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X,
    Y,
    test_size=0.4,
    random_state=42
)

X_Val, X_test, Y_Val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.5,
    random_state=42
)

print("Training shape:", X_train.shape, Y_train.shape)
print("Validation shape:", X_Val.shape, Y_Val.shape)
print("Test shape:", X_test.shape, Y_test.shape)

# Fit the model using multiple random initializations

model = MultivariateLinearRegressionGD(
  learning_rate=1e-5,
  max_iter=10000,
  tolerance=1e-8,
  n_initializations=10,
  random_state=42
)

model.fit(X_train, X_Val, Y_train, Y_Val)

model.report_validation_performance()

# Compare different learning rates

learning_rates = [1e-7, 5e-7, 1e-6, 2e-6]

for lr in learning_rates:
    model = MultivariateLinearRegressionGD(
        learning_rate=lr,
        max_iter=10000,
        tolerance=1e-8,
        n_initializations=10,
        random_state=42
    )

    model.fit(X_train, X_Val, Y_train, Y_Val)

    print(
        f"Learning rate: {lr}, "
        f"Best validation MSE: "
        f"{model.validation_loss_history[model.selected_initialization]:.6f}"
    )

# Compare different numbers of iterations

max_iterations = [5000, 10000, 20000, 50000]

for max_iter in max_iterations:
    model = MultivariateLinearRegressionGD(
        learning_rate=1e-7,
        max_iter=max_iter,
        tolerance=1e-8,
        n_initializations=10,
        random_state=42
    )

    model.fit(X_train, X_Val, Y_train, Y_Val)

    best_mse = model.validation_loss_history[model.selected_initialization]

    print(f"Max iterations: {max_iter}, Best validation MSE: {best_mse:.6f}")

# Compare different convergence tolerances

tolerances = [1e-4, 1e-6, 1e-8, 1e-10]

for tol in tolerances:
    model = MultivariateLinearRegressionGD(
        learning_rate=1e-7,
        max_iter=5000,
        tolerance=tol,
        n_initializations=10,
        random_state=42
    )

    model.fit(X_train, X_Val, Y_train, Y_Val)

    best_mse = model.validation_loss_history[model.selected_initialization]

    print(f"Tolerance: {tol}, Best validation MSE: {best_mse:.6f}")




Training shape: (12, 3) (12, 3)
Validation shape: (4, 3) (4, 3)
Test shape: (4, 3) (4, 3)
Initialization 1: Validation MSE = 1199.6410738620
Initialization 2: Validation MSE = 1199.5222721713
Initialization 3: Validation MSE = 1199.6851457379
Initialization 4: Validation MSE = 1199.7218883072
Initialization 5: Validation MSE = 1199.5668694107
Initialization 6: Validation MSE = 1199.8569960136
Initialization 7: Validation MSE = 1199.7446097223
Initialization 8: Validation MSE = 1199.7541212938
Initialization 9: Validation MSE = 1199.7287979770
Initialization 10: Validation MSE = 1199.5744417535

Selected Initialization: 2 with Validation MSE = 1199.5222721713
Learning rate: 1e-07, Best validation MSE: 1046.691159
Learning rate: 5e-07, Best validation MSE: 1104.553828
Learning rate: 1e-06, Best validation MSE: 1120.579111
Learning rate: 2e-06, Best validation MSE: 1147.854289
Max iterations: 5000, Best validation MSE: 944.799408
Max iterations: 10000, Best validation MSE: 1046.691159
Max

## Final model using selected hyperparameters

In [34]:


final_model = MultivariateLinearRegressionGD(
    learning_rate=1e-7,
    max_iter=5000,
    tolerance=1e-8,
    n_initializations=10,
    random_state=42
)

final_model.fit(X_train, X_Val, Y_train, Y_Val)

# Report validation MSE for every initialization
final_model.report_validation_performance()

# Final evaluation

Y_train_pred = final_model.predict(X_train)
Y_Val_pred = final_model.predict(X_Val)
Y_test_pred = final_model.predict(X_test)

train_mse = np.mean((Y_train_pred - Y_train) ** 2)
validation_mse = np.mean((Y_Val_pred - Y_Val) ** 2)
test_mse = np.mean((Y_test_pred - Y_test) ** 2)

print("\nFinal Model Performance:")
print(f"Training MSE: {train_mse:.6f}")
print(f"Validation MSE: {validation_mse:.6f}")
print(f"Test MSE: {test_mse:.6f}")

Initialization 1: Validation MSE = 947.3292291517
Initialization 2: Validation MSE = 946.6987297174
Initialization 3: Validation MSE = 948.6118390832
Initialization 4: Validation MSE = 945.9313138007
Initialization 5: Validation MSE = 944.7994077501
Initialization 6: Validation MSE = 947.2665604197
Initialization 7: Validation MSE = 947.9628384418
Initialization 8: Validation MSE = 947.6199107928
Initialization 9: Validation MSE = 946.5183490772
Initialization 10: Validation MSE = 945.6879480659

Selected Initialization: 5 with Validation MSE = 944.7994077501

Final Model Performance:
Training MSE: 3350.791533
Validation MSE: 944.799408
Test MSE: 1554.905374


## Problem 5 Discussion 

The model had a training MSE of 3350.791533, a validation MSE of 944.799408, and a test MSE of 1554.905374. The test MSE was higher than the validation MSE, showing that the model performed somewhat better on the validation data than on the unseen test data.

For optimization, a learning rate of $1\times10^{-7}$ and 5,000 iterations produced the best validation performance among the tested settings. Larger learning rates caused the optimization to become unstable.

Using multiple random initializations produced similar validation MSE values. Initialization 5 performed the best with a validation MSE of 944.799408, showing that the choice of starting weights had a relatively small effect on the final result.

The main limitation is the very small dataset of only 20 observations. With so few observations, the training, validation, and test results can vary significantly depending on how the data is split, making it difficult to draw strong conclusions about how well the model would perform on new data.